# 05. Attention from Scratch

## 学習目標

- causal self-attention を**手計算**（3トークン・2次元）で1ステップずつ再現する
- 自作 explicit 実装と PyTorch SDPA の**出力が一致**することを確認する（速度は別物）
- attention 重みの性質（行和=1・未来ゼロ・エントロピー）を測る

## 数式（1 head 分）

$$Q = XW_Q,\quad K = XW_K,\quad V = XW_V \qquad X \in \mathbb{R}^{T \times d}$$

$$\text{scores} = \frac{QK^\top}{\sqrt{d_h}} \in \mathbb{R}^{T\times T},\qquad
\text{scores}_{tj} \leftarrow -\infty \;\; (j > t \text{: causal mask})$$

$$A = \operatorname{softmax}_{\text{row}}(\text{scores}),\qquad \text{out} = AV$$

$\sqrt{d_h}$ で割るのは、次元が増えても内積の分散を ~1 に保ち softmax の飽和を防ぐため。

In [1]:
# 共通セットアップ（全ノートブック同一）
import warnings
warnings.filterwarnings("ignore")

import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT={ROOT}")
print(f"device={DEVICE}, torch={torch.__version__}")

ROOT=/home/kazumasa/projects/jp_llm_lab
device=cuda, torch=2.11.0+cu128


In [2]:
# 手計算例: T=3, d_h=2 の小さな数値で全ステップを表示
np.set_printoptions(precision=3, suppress=True)
Q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
V = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, 2.0]])

scores = Q @ K.T / np.sqrt(2)
print("scores = QKᵀ/√d_h =")
print(scores)

mask = np.triu(np.ones((3, 3), dtype=bool), k=1)
masked = np.where(mask, -np.inf, scores)
print("\ncausal mask 適用（未来 j>t を -inf に）=")
print(masked)

weights = np.exp(masked) / np.exp(masked).sum(axis=1, keepdims=True)
print("\nsoftmax（行ごとに正規化）=")
print(weights)
print("行和:", weights.sum(axis=1))

out = weights @ V
print("\nout = A V =")
print(out)

scores = QKᵀ/√d_h =
[[0.707 0.707 0.   ]
 [0.    0.707 0.707]
 [0.707 1.414 0.707]]

causal mask 適用（未来 j>t を -inf に）=
[[0.707  -inf  -inf]
 [0.    0.707  -inf]
 [0.707 1.414 0.707]]

softmax（行ごとに正規化）=
[[1.    0.    0.   ]
 [0.33  0.67  0.   ]
 [0.248 0.503 0.248]]
行和: [1. 1. 1.]

out = A V =
[[1.    0.   ]
 [0.33  0.67 ]
 [0.745 1.   ]]


**読み解き**:
- 1行目（t=0）は自分しか見えない → 重み `[1, 0, 0]`、出力は $v_0$ そのもの。
- 2行目（t=1）は $q_1 k_0^\top vs q_1 k_1^\top$ の大小で過去2トークンを按分。
- $-\infty$ は $e^{-\infty}=0$ となり、**softmax の分母からも消える** — 「見えない」の実装。

In [3]:
# 自作モジュールの explicit パスを、手動再計算で完全一致検証
from jp_llm_lab.models.attention import CausalSelfAttention

torch.manual_seed(0)
attn = CausalSelfAttention(d_model=8, n_heads=2, context_len=8, attn_impl="explicit")
x = torch.randn(1, 4, 8)
t = {}
y = attn(x, trace=t, prefix="a")

q, k, v = attn.qkv(x).split(8, dim=2)
q = q.view(1, 4, 2, 4).transpose(1, 2)
k = k.view(1, 4, 2, 4).transpose(1, 2)
v = v.view(1, 4, 2, 4).transpose(1, 2)
scores = q @ k.transpose(-2, -1) / math.sqrt(4)
scores = scores.masked_fill(~torch.tril(torch.ones(4, 4, dtype=torch.bool)), float("-inf"))
w = scores.softmax(-1)
y_manual = attn.proj((w @ v).transpose(1, 2).reshape(1, 4, 8))

assert torch.allclose(w, t["a.attn_weights"], atol=1e-6)
assert torch.allclose(y_manual, y, atol=1e-6)
print("手動再計算 == モジュール出力 ✓")
print("重み行和（全head×全行）:", w.sum(-1).flatten().tolist())

手動再計算 == モジュール出力 ✓
重み行和（全head×全行）: [1.0, 1.0, 1.0, 0.9999999403953552, 1.0, 1.0, 1.0, 0.9999999403953552]


In [4]:
# causal mask の可視化
T = 8
mask = torch.tril(torch.ones(T, T))
fig = go.Figure(go.Heatmap(z=mask.tolist(), colorscale=[[0, "#eeeeee"], [1, "#3b4fd8"]],
                           showscale=False))
fig.update_layout(title="causal mask（青=参照可 / 灰=遮断） 行=query位置, 列=key位置",
                  template="plotly_white", height=380, width=420)
fig.update_yaxes(autorange="reversed", title="query t")
fig.update_xaxes(title="key j")
fig.show()

In [5]:
# explicit と SDPA の出力一致（教育実装は「遅いが同じ計算」であることの確認）
import torch.nn.functional as F
torch.manual_seed(1)
attn2 = CausalSelfAttention(64, 4, 32, attn_impl="explicit")
x2 = torch.randn(2, 16, 64)
y_explicit = attn2(x2)
attn2.set_attn_impl("sdpa")
y_sdpa = attn2(x2)
print("max |explicit - sdpa| =", float((y_explicit - y_sdpa).abs().max()))
assert torch.allclose(y_explicit, y_sdpa, atol=1e-5)
print("一致 ✓（同一パラメータ・同一数式・異なるカーネル）")

max |explicit - sdpa| = 1.1920928955078125e-07
一致 ✓（同一パラメータ・同一数式・異なるカーネル）


In [6]:
# 実測ベンチ（scripts/bench_attention.py の保存結果）
from jp_llm_lab.visualization.params_viz import attn_bench_figure
bench = load_json(ROOT / "reports/figures/attn_bench.json")
fig = attn_bench_figure(bench)
fig.show()
sdpa_512 = [r for r in bench["results"] if r["T"] == 512 and r["dtype"] == "bfloat16"]
if len(sdpa_512) == 2:
    e, s = (r["median_ms"] for r in sdpa_512)
    print(f"T=512 bf16: explicit {e:.2f}ms vs SDPA {s:.2f}ms → ×{e/s:.1f}")

T=512 bf16: explicit 7.45ms vs SDPA 1.13ms → ×6.6


In [7]:
# 学習済みモデルの attention エントロピー（init vs trained）
from jp_llm_lab.models.config import ModelConfig
from jp_llm_lab.models.transformer import ClassicalGPT
from jp_llm_lab.training.trainer import load_checkpoint
from jp_llm_lab.tokenization.char_tokenizer import CharTokenizer
from jp_llm_lab.visualization.attention_viz import attention_entropy, entropy_comparison_figure

RUN = ROOT / "experiments/runs/m1_model_s_smoke_seed42"
assert RUN.exists(), "先に `make -C jp_llm_lab train-smoke` を実行してください"
tok_run = CharTokenizer.load(RUN / "tokenizer.json")

def load_model(pattern):
    p = sorted(RUN.glob(f"checkpoints/{pattern}"))[0]
    payload = torch.load(p, map_location="cpu", weights_only=False)
    m = ClassicalGPT(ModelConfig.from_dict(payload["model_cfg"]))
    load_checkpoint(p, m)
    return m.eval()

s = "私はその人を常に先生と呼んでいた。"
ids_s = torch.tensor([tok_run.encode(s)])
ent = {}
for label, m in [("init", load_model("ckpt_000pct_*.pt")), ("trained", load_model("ckpt_100pct_*.pt"))]:
    maps = m.attention_maps(ids_s)
    ent[label] = torch.stack([attention_entropy(w) for w in maps])
    print(f"{label:8s} 平均エントロピー: {float(ent[label].mean()):.3f} nats"
          f"  層別: {[round(float(e.mean()), 2) for e in ent[label]]}")
fig = entropy_comparison_figure(ent, n_layers=4, n_heads=4)
fig.show()

init     平均エントロピー: 1.970 nats  層別: [1.97, 1.97, 1.97, 1.97]
trained  平均エントロピー: 1.688 nats  層別: [1.85, 1.7, 1.72, 1.48]


## Observation / Interpretation / Caveat

- **Observation**: init では全 head のエントロピーがほぼ同一（≈1.97 nats、一様に近い）。
  学習後は平均が低下し、**最深層 layer 3 が最も選択的**。head 間のばらつきも学習後に生じる。
- **Interpretation**: 学習は「どこを見るか」を一様から特定パターンへ分化させる。
  ただしこの文では「直前トークン注視が支配的になる」という素朴な予想は部分的にしか成り立たず、
  特定の内容文字への集中が主だった（HTMLレポートの heatmap 参照）。
- **Caveat（重要）**: **attention 重み ≠ 説明**。重みが大きくても出力への因果的寄与が大きいとは
  限らない（値ベクトル $v_j$ が小さければ寄与も小さい）。因果性の検証には ablation /
  activation patching が必要（M2 §9 で実施）。
  また、行 $t$ のエントロピー上限は $\ln(t{+}1)$ なので、短い文の平均値は構造的に小さく出る。

## 確認問題

1. $\sqrt{d_h}$ スケーリングを外すと softmax はどうなるか（大きい $d_h$ で）。
2. mask を softmax の**後**に掛けると何が壊れるか。
3. SDPA が explicit より速い主因はメモリと計算のどちらか。

**次へ**: [06_transformer_forward_pass](06_transformer_forward_pass.ipynb)